# 05 - Interactive Custom Sneaker Design

This notebook provides an interactive design panel for latent-space editing.

## Features
- Sliders for `Dim0` to `Dim15` (or up to model latent dimension).
- Numeric input boxes synced with each slider.
- One-click `Generate Design` button.
- Optional `Reset` and `Randomize` buttons.

The generated result is decoded directly by the trained Beta-VAE decoder.

In [16]:

import os
import warnings

# Silence Python-level warnings in notebook output.
warnings.filterwarnings('ignore')

# Reduce noisy runtime warnings from OpenMP/MKL in some environments.
os.environ['KMP_WARNINGS'] = '0'
os.environ['MKL_VERBOSE'] = '0'

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import ipywidgets as widgets
from IPython.display import display, clear_output


In [17]:

def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / 'processed_sneakers_64x64').exists() or (cwd / 'BayesML_Final_Project.ipynb').exists():
        return cwd
    if (cwd / '..' / 'processed_sneakers_64x64').resolve().exists() or (cwd / '..' / 'BayesML_Final_Project.ipynb').resolve().exists():
        return (cwd / '..').resolve()
    return cwd


PROJECT_ROOT = resolve_project_root()
PROJECT_ROOT


PosixPath('/Users/hjjh/Desktop/ML Project/Sneaker-Design-VAE')

## Paths and Model Loading

In [18]:

CHECKPOINT_DIR = PROJECT_ROOT / 'artifacts' / 'checkpoints'
GENERATED_DIR = PROJECT_ROOT / 'artifacts' / 'generated'
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

IMPROVED_CHECKPOINT_PATH = CHECKPOINT_DIR / 'beta_vae_improved.pt'
BASELINE_CHECKPOINT_PATH = CHECKPOINT_DIR / 'beta_vae_baseline.pt'

LATENT_DIM_DEFAULT = 16

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Using device: {device}')


Using device: mps


In [19]:

class BetaVAE(nn.Module):
    def __init__(self, latent_dim: int = 16):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
        )

        self.fc_mu = nn.Linear(4096, latent_dim)
        self.fc_log_var = nn.Linear(4096, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 4096)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def reparameterize(self, mu: torch.Tensor, log_var: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x: torch.Tensor):
        encoded = self.encoder(x)
        flat = torch.flatten(encoded, start_dim=1)
        mu = self.fc_mu(flat)
        log_var = self.fc_log_var(flat)
        z = self.reparameterize(mu, log_var)

        decoded = self.fc_decode(z)
        decoded = decoded.view(-1, 256, 4, 4)
        reconstruction = self.decoder(decoded)
        return reconstruction, mu, log_var


if IMPROVED_CHECKPOINT_PATH.exists():
    checkpoint_path = IMPROVED_CHECKPOINT_PATH
elif BASELINE_CHECKPOINT_PATH.exists():
    checkpoint_path = BASELINE_CHECKPOINT_PATH
else:
    raise FileNotFoundError(
        'No trained checkpoint found. Run 02_baseline_training.ipynb and/or '
        '03_improved_training_with_annealing.ipynb first.'
    )

checkpoint = torch.load(checkpoint_path, map_location=device)
latent_dim = checkpoint.get('latent_dim', LATENT_DIM_DEFAULT)
model = BetaVAE(latent_dim=latent_dim).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f'Loaded checkpoint: {checkpoint_path}')
print(f'Latent dim: {latent_dim}')


Loaded checkpoint: /Users/hjjh/Desktop/ML Project/Sneaker-Design-VAE/artifacts/checkpoints/beta_vae_improved.pt
Latent dim: 16


## Interactive Design Panel (`Dim0` to `Dim15`)

In [20]:
def decode_from_latent(model: nn.Module, latent_vector: torch.Tensor) -> np.ndarray:
    with torch.no_grad():
        decoded = model.fc_decode(latent_vector)
        decoded = decoded.view(-1, 256, 4, 4)
        generated = model.decoder(decoded)
    image = generated[0].detach().cpu().numpy()
    image = np.transpose(image, (1, 2, 0))
    return np.clip(image, 0.0, 1.0)


def build_interactive_designer(
    model: nn.Module,
    device: torch.device,
    latent_dim: int,
    controlled_dims_count: int = None,
):
    if controlled_dims_count is None:
        controlled_dims_count = min(latent_dim, 16)

    controlled_dims = list(range(controlled_dims_count))
    slider_pairs = []

    # Build one row per dimension: label + slider + numeric input.
    for dim in controlled_dims:
        slider = widgets.FloatSlider(
            value=0.0,
            min=-3.5,
            max=3.5,
            step=0.1,
            description='',
            readout=False,
            continuous_update=False,
            layout=widgets.Layout(width='420px'),
        )
        value_box = widgets.FloatText(
            value=0.0,
            step=0.1,
            layout=widgets.Layout(width='95px'),
        )
        widgets.jslink((slider, 'value'), (value_box, 'value'))
        label = widgets.HTML(value=f'<b>Dim {dim}</b>')
        row = widgets.HBox([label, slider, value_box])
        slider_pairs.append((slider, value_box, row))

    generate_btn = widgets.Button(description='Generate Design', button_style='success')
    reset_btn = widgets.Button(description='Reset', button_style='warning')
    random_btn = widgets.Button(description='Randomize', button_style='info')
    save_btn = widgets.Button(description='Save PNG', button_style='')

    status = widgets.HTML(value='Ready. Adjust values and click <b>Generate Design</b>.')
    output = widgets.Output()

    def current_dim_values():
        return {dim: slider_pairs[i][0].value for i, dim in enumerate(controlled_dims)}

    def render_image(values_dict, save=False):
        z = torch.zeros(1, latent_dim, device=device)
        for dim, value in values_dict.items():
            if dim < latent_dim:
                z[0, dim] = float(value)

        image = decode_from_latent(model, z)
        with output:
            clear_output(wait=True)
            plt.figure(figsize=(4.8, 4.8))
            plt.imshow(image)
            plt.axis('off')
            title_str = ', '.join([f'D{d}={v:.1f}' for d, v in values_dict.items()])
            plt.title('Custom Sneaker\n' + title_str, fontsize=10)
            plt.show()

        if save:
            output_path = GENERATED_DIR / 'custom_sneaker_interactive.png'
            plt.imsave(output_path, image)
            status.value = f'Saved image to: <code>{output_path}</code>'
        else:
            status.value = 'Design generated successfully.'

    def on_generate_clicked(_):
        render_image(current_dim_values(), save=False)

    def on_reset_clicked(_):
        for slider, value_box, _row in slider_pairs:
            slider.value = 0.0
            value_box.value = 0.0
        render_image(current_dim_values(), save=False)

    def on_random_clicked(_):
        for slider, _value_box, _row in slider_pairs:
            slider.value = np.random.uniform(-3.0, 3.0)
        render_image(current_dim_values(), save=False)

    def on_save_clicked(_):
        render_image(current_dim_values(), save=True)

    generate_btn.on_click(on_generate_clicked)
    reset_btn.on_click(on_reset_clicked)
    random_btn.on_click(on_random_clicked)
    save_btn.on_click(on_save_clicked)

    controls = [row for _, _, row in slider_pairs]
    action_row = widgets.HBox([generate_btn, reset_btn, random_btn, save_btn])
    dims_text = widgets.HTML(value=f'Controlling dimensions: <b>Dim0</b> to <b>Dim{controlled_dims_count - 1}</b>')
    ui = widgets.VBox([dims_text] + controls + [action_row, status, output])

    display(ui)
    render_image(current_dim_values(), save=False)

In [21]:
build_interactive_designer(
    model=model,
    device=device,
    latent_dim=latent_dim,
    controlled_dims_count=min(latent_dim, 16),
)